## **Connect Colab**

In [ ]:
import os, sys, subprocess
from getpass import getpass

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "real-video"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "KEY"

## **Download DATASET**

In [ ]:
from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
!pip install -q yt-dlp

In [ ]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

## **Setup (30 classes)**

Switch the class configuration to the full 10 + 10 + 10 stream, in the notebook, before anything is built.

In [ ]:
cfg.TASK_1 = ["ApplyEyeMakeup", "ApplyLipstick", "Archery", "BabyCrawling", "BalanceBeam",
              "BandMarching", "BlowDryHair", "BlowingCandles", "BodyWeightSquats", "Bowling"]
cfg.TASK_2 = ["BoxingPunchingBag", "BoxingSpeedBag", "BrushingTeeth", "CliffDiving", "CricketBowling",
              "CricketShot", "CuttingInKitchen", "FieldHockeyPenalty", "Haircut", "SoccerPenalty"]

In [ ]:
ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2
class_to_idx     = {cls_name: i for i, cls_name in enumerate(ALL_CLASSES_LIST)}
n_base = len(cfg.SELECTED_CLASSES)
n_t1   = len(cfg.TASK_1)
n_t2   = len(cfg.TASK_2)
print(f"base {n_base}  task1 {n_t1}  task2 {n_t2}  total {len(ALL_CLASSES_LIST)}")

## **Preprocess data**

In [ ]:
base_split  = build_group_split(cfg.SELECTED_CLASSES, train_groups=18, val_groups=3)
task1_split = build_group_split(cfg.TASK_1,           train_groups=18, val_groups=3)
task2_split = build_group_split(cfg.TASK_2,           train_groups=18, val_groups=3)

In [ ]:
# BASE  (max_samples caps TRAIN clips/class via cfg.MAX_SAMPLES; None = full data, val/test untouched)
preprocess_group_split(base_split, cfg.SELECTED_CLASSES, output_root=cfg.BASE_ROOT, max_samples=cfg.MAX_SAMPLES)

In [ ]:
# TASK 1  (same TRAIN-only cap via cfg.MAX_SAMPLES)
preprocess_group_split(task1_split, cfg.TASK_1, output_root=cfg.TASK1_ROOT, max_samples=cfg.MAX_SAMPLES)

In [ ]:
# TASK 2  (same TRAIN-only cap via cfg.MAX_SAMPLES)
preprocess_group_split(task2_split, cfg.TASK_2, output_root=cfg.TASK2_ROOT, max_samples=cfg.MAX_SAMPLES)

## **Create Loaders**

In [ ]:
base_train  = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "train"), class_to_idx=class_to_idx)
base_val    = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "val"),   class_to_idx=class_to_idx)
base_test   = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "test"),  class_to_idx=class_to_idx)
task1_train = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "train"), class_to_idx=class_to_idx)
task1_val   = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "val"),   class_to_idx=class_to_idx)
task1_test  = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "test"),  class_to_idx=class_to_idx)
task2_train = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "train"), class_to_idx=class_to_idx)
task2_val   = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "val"),   class_to_idx=class_to_idx)
task2_test  = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "test"),  class_to_idx=class_to_idx)

base_train_loader = DataLoader(base_train, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
base_val_loader   = DataLoader(base_val,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
base_test_loader  = DataLoader(base_test,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
task1_train_loader = DataLoader(task1_train, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
task1_val_loader   = DataLoader(task1_val,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
task1_test_loader  = DataLoader(task1_test,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
task2_train_loader = DataLoader(task2_train, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
task2_val_loader   = DataLoader(task2_val,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
task2_test_loader  = DataLoader(task2_test,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

combined_test_loader  = DataLoader(ConcatDataset([base_test, task1_test]),              batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
combined_test_loader2 = DataLoader(ConcatDataset([base_test, task1_test, task2_test]),  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

print(f"base {len(base_train)}/{len(base_val)}/{len(base_test)}  "
      f"task1 {len(task1_train)}/{len(task1_val)}/{len(task1_test)}  "
      f"task2 {len(task2_train)}/{len(task2_val)}/{len(task2_test)}")

## **Backbone: ResNet50 on base**

Train the ResNet50 + LSTM on the 10 base classes - the teacher every later stage builds on.

In [ ]:
teacher = ResNet50LSTM(num_classes=n_base).to(device)
set_seed(cfg.SEED)
hist_base = train_model(teacher, base_train_loader, base_val_loader, num_epochs=cfg.CL_EPOCHS, lr=cfg.CL_LR, device=device)

In [ ]:
plot_training_results(hist_base, task_name="ResNet50 base")

In [ ]:
base_acc, base_preds, base_labels = test_model(teacher, base_test_loader, device)
plot_confusion_matrix(base_labels, base_preds, num_classes=n_base,
                      classes_list=cfg.SELECTED_CLASSES, name="ResNet50 base")

## **Continual Learning: Replay only (rehearsal)**

Grow the head base -> 20 -> 30 with a replay buffer only (no LwF) - the balanced continual-learning method from stage 3.

In [ ]:
model = unfreeze_all(teacher)
snapshots = []
forget_hist = []

In [ ]:
buffer = ReplayBuffer(max_size=600)
fill_buffer(buffer, base_train_loader, per_class=15)
print(f"buffer: {len(buffer.data)} clips")

In [ ]:
model = expand_classifier(model, n_base + n_t1)
model = unfreeze_all(model).to(device)
snapshots.append(evaluate_all_tasks(model, device, base_test_loader, [task1_test_loader], [combined_test_loader]))

In [ ]:
set_seed(cfg.SEED)
track1 = {"Base": base_test_loader, "Task 1": task1_test_loader}
train_t1 = train_continual(model, task1_train_loader, task1_val_loader, device,
                           buffer=buffer, teacher=None, num_old_classes=n_base,
                           epochs=cfg.CL_EPOCHS, lr=cfg.CL_LR,
                           track_loaders=track1, track_history=forget_hist)
torch.cuda.empty_cache()

In [ ]:
plot_training_results(train_t1, task_name="Task 1 - Replay")
snapshots.append(evaluate_all_tasks(model, device, base_test_loader, [task1_test_loader], [combined_test_loader]))

In [ ]:
fill_buffer(buffer, task1_train_loader, per_class=15)
model = expand_classifier(model, n_base + n_t1 + n_t2)
model = unfreeze_all(model).to(device)
print(f"buffer: {len(buffer.data)} clips  |  head out={model.fc.out_features}")

In [ ]:
set_seed(cfg.SEED)
track2 = {"Base": base_test_loader, "Task 1": task1_test_loader, "Task 2": task2_test_loader}
train_t2 = train_continual(model, task2_train_loader, task2_val_loader, device,
                           buffer=buffer, teacher=None, num_old_classes=n_base + n_t1,
                           epochs=cfg.CL_EPOCHS, lr=cfg.CL_LR,
                           track_loaders=track2, track_history=forget_hist)
torch.cuda.empty_cache()

In [ ]:
plot_training_results(train_t2, task_name="Task 2 - Replay")
snapshots.append(evaluate_all_tasks(model, device, base_test_loader,
                                    [task1_test_loader, task2_test_loader],
                                    [combined_test_loader, combined_test_loader2]))

In [ ]:
cl_acc, cl_preds, cl_labels = test_model(model, combined_test_loader2, device)
plot_confusion_matrix(cl_labels, cl_preds, num_classes=len(ALL_CLASSES_LIST),
                      classes_list=ALL_CLASSES_LIST, name="Continual (30 classes)")

In [ ]:
plot_forgetting_history(forget_hist, epochs_per_task=cfg.CL_EPOCHS)

## **Distillation: CE + Cosine (best KD from the study)**

Distil the 30-class replay model into a MobileNet student with feature-cosine KD - the best variant in stage 4 (~0.83).

In [ ]:
train_all_loader = DataLoader(ConcatDataset([base_train, task1_train, task2_train]), batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
val_all_loader   = DataLoader(ConcatDataset([base_val,   task1_val,   task2_val]),   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
student = MobileNetV3SmallLSTMStudent(num_classes=len(ALL_CLASSES_LIST)).to(device)
set_seed(cfg.SEED)
student, hist_kd = train_student(student, model, train_all_loader, val_all_loader, device,
                                 mode="cosine", T=cfg.KD_T, ce_weight=cfg.KD_CE_WEIGHT,
                                 distill_weight=cfg.KD_DISTILL_WEIGHT,
                                 epochs=cfg.KD_EPOCHS, lr=cfg.KD_LR)
torch.cuda.empty_cache()

In [ ]:
plot_training_results(hist_kd, task_name="student CE + Cosine")

In [ ]:
student_acc, student_preds, student_labels = test_model(student, combined_test_loader2, device)
plot_confusion_matrix(student_labels, student_preds, num_classes=len(ALL_CLASSES_LIST),
                      classes_list=ALL_CLASSES_LIST, name="Student CE + Cosine")

In [ ]:
teacher_params = sum(p.numel() for p in model.parameters())
student_params = sum(p.numel() for p in student.parameters())
print(f"continual model : {cl_acc:.4f}  ({teacher_params/1e6:.1f}M)")
print(f"student (cosine)  : {student_acc:.4f}  ({student_params/1e6:.1f}M, {teacher_params/student_params:.0f}x smaller)")

## **MAML on the student (MAML + replay, best from the meta study)**

Meta-train the distilled student on NOVEL sports (disjoint from the 30) so it learns *how to adapt fast*. This is the meta-init we use to pick up new / real-video classes from a few clips.

In [ ]:
# novel meta classes (not among the 30) - kept small for a fast final-pipeline meta-train
known      = set(ALL_CLASSES_LIST)
MAML_META  = [c for c in cfg.SPORTS_META if c not in known][:15]
maml_c2i   = {c: i for i, c in enumerate(ALL_CLASSES_LIST + MAML_META)}
ULAL_META_ROOT = f"{cfg.DATA_ROOT}/grouped_ulal_meta"
meta_split = build_group_split(MAML_META, train_groups=18, val_groups=3)
preprocess_group_split(meta_split, MAML_META, output_root=ULAL_META_ROOT, max_samples=8, max_test_samples=6)
meta_train_ds = UCF101Clips(os.path.join(ULAL_META_ROOT, "train"), class_to_idx=maml_c2i)
pool_clips, pool_labels = build_pool([meta_train_ds], per_class=6)
meta_ids = sorted(set(pool_labels.tolist()))
print(f"MAML meta classes {len(MAML_META)} | pool {tuple(pool_clips.shape)}")

In [ ]:
set_seed(cfg.SEED)
meta_student_model = copy.deepcopy(student)
meta_student_model.fc = nn.Linear(cfg.HIDDEN_SIZE, 5).to(device)    # throwaway 5-way episode head
# meta-train only the last blocks (base features kept), then re-fit the 30-way head to the shifted features
meta_student_model, hist_maml = meta_train_maml(meta_student_model, pool_clips, pool_labels, meta_ids,
                                                5, 3, 3, 6, 40, 0.05, 5, 1e-4, device,
                                                freeze_backbone=True, train_last=4)
meta_student_model = recalibrate_head(meta_student_model, train_all_loader, len(ALL_CLASSES_LIST), device)
del pool_clips, pool_labels; gc.collect(); torch.cuda.empty_cache()

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(hist_maml["query_accs"]) + 1), hist_maml["query_accs"], marker="s")
plt.xlabel("meta epoch"); plt.ylabel("meta query accuracy"); plt.ylim(0, 1.05)
plt.title("MAML meta-training on the student (novel sports)")
plt.tight_layout(); plt.show()

In [ ]:
# MAML student on the 30 known classes + confusion matrix. Meta-training shifts the backbone, so
# base-30 usually DROPS a bit - the meta-student trades some base for fast adaptability to NEW classes.
maml_acc, maml_preds, maml_labels = test_model(meta_student_model, combined_test_loader2, device)
print(f"MAML student on 30 classes: {maml_acc:.4f}  (KD student was {student_acc:.4f})")
plot_confusion_matrix(maml_labels, maml_preds, num_classes=len(ALL_CLASSES_LIST),
                      classes_list=ALL_CLASSES_LIST, name="MAML student (30 classes)")
torch.cuda.empty_cache()

## **YouTube real-video test (domain robustness + learn-from-few)**

Two things on real videos: (a) KNOWN classes - does the student still recognise them out of the UCF domain? (b) NEW classes (not in the 30) - can the MAML student learn them from a few real clips? A model has no output slot for a brand-new class, so BEFORE adapting it scores ~0; a few-shot MAML adaptation gives it a slot. Long videos yield many clips (enough for support + query); short ones are the harder stress test.

NOTE: swap in your own URLs below (mix long + short). YouTube on Colab can be bot-blocked - archive.org links or an uploaded file are more reliable.

In [ ]:
YT_VIDEOS = {
    # KNOWN (already in the 30) -> domain robustness, tested directly (no adaptation)
    "PushUps": ["https://www.youtube.com/watch?v=f9TERHtc1LA"],
    "Bowling": ["https://www.youtube.com/watch?v=REPLACE_ME"],
    # NEW (NOT in the 30) -> MAML learn-from-few (give a LONG video so there are enough clips)
    "Skiing":  ["https://www.youtube.com/watch?v=REPLACE_ME_LONG"],
    "Surfing": ["https://www.youtube.com/watch?v=REPLACE_ME"],
}
yt_known = [c for c in YT_VIDEOS if c in set(ALL_CLASSES_LIST)]
yt_new   = [c for c in YT_VIDEOS if c not in set(ALL_CLASSES_LIST)]
yt_c2i   = dict(class_to_idx)
for c in yt_new:
    yt_c2i[c] = len(yt_c2i)
yt_clips, yt_labels, yt_videos = download_youtube_clips(YT_VIDEOS, yt_c2i, clips_per_video=20)
print(f"clips {len(yt_clips)} | known {yt_known} | new {yt_new}")
torch.cuda.empty_cache()

In [ ]:
UCF101Clips.show_frame(yt_clips, yt_labels, clip_idx=0, frame_idx=0)
UCF101Clips.show_frame(yt_clips, yt_labels, clip_idx=len(yt_clips) - 1, frame_idx=0)

In [ ]:
# (a) KNOWN classes - robustness of the student on real YouTube clips (no adaptation)
n30 = len(ALL_CLASSES_LIST)
kmask = yt_labels < n30
if int(kmask.sum()) > 0:
    kloader = DataLoader(TensorDataset(yt_clips[kmask], yt_labels[kmask]), batch_size=cfg.BATCH_SIZE)
    yt_known_acc, kp, kl = test_model(student, kloader, device)
    print(f"KNOWN-class robustness (student on real YouTube): {yt_known_acc:.3f} over {int(kmask.sum())} clips")
    plot_confusion_matrix(kl, kp, num_classes=len(ALL_CLASSES_LIST),
                          classes_list=ALL_CLASSES_LIST, name="YouTube KNOWN classes (real video)")
else:
    yt_known_acc = float('nan'); print("no known-class clips downloaded")
torch.cuda.empty_cache()

In [ ]:
# (b) NEW classes - MAML learns them from a few real clips (before = no slot -> ~0; after = adapted)
nmask = yt_labels >= n30
n_yt_new = len(yt_new)
yt_new_before = yt_new_after = float('nan')
if int(nmask.sum()) > 0 and n_yt_new > 0:
    nx, ny = yt_clips[nmask], yt_labels[nmask]
    sx_l, sy_l, qx_l, qy_l = [], [], [], []
    for c in ny.unique():
        idx = (ny == c).nonzero(as_tuple=True)[0]
        idx = idx[torch.randperm(len(idx))]
        sx_l.append(nx[idx[:5]]); sy_l += [int(c)] * min(5, len(idx))
        qx_l.append(nx[idx[5:]]); qy_l += [int(c)] * max(0, len(idx) - 5)
    support_x = torch.cat(sx_l).to(device); support_y = torch.tensor(sy_l).to(device)
    qloader   = DataLoader(TensorDataset(torch.cat(qx_l), torch.tensor(qy_l)), batch_size=cfg.BATCH_SIZE)
    m = expand_classifier(copy.deepcopy(meta_student_model), n30 + n_yt_new).to(device)
    yt_new_before, _, _ = test_model(m, qloader, device)
    set_seed(cfg.SEED)
    m_adapt = few_shot_adapt(m, support_x, support_y, 0.01, 20)
    yt_new_after, _, _  = test_model(m_adapt, qloader, device)
    print(f"NEW-class learn-from-few (MAML student): before {yt_new_before:.3f} -> after {yt_new_after:.3f}")
    del m, m_adapt; gc.collect(); torch.cuda.empty_cache()
else:
    print("no new-class clips downloaded")

## **Summary**

In [ ]:
teacher_params = sum(p.numel() for p in model.parameters())
student_params = sum(p.numel() for p in student.parameters())
print("ULAL pipeline")
print(f"  ResNet50 base (10)         : {base_acc:.4f}")
print(f"  Replay CL (30, all)        : {cl_acc:.4f}")
print(f"  MobileNet student (cosine) : {student_acc:.4f}  ({teacher_params/student_params:.0f}x smaller)")
print(f"  MAML student on 30 classes : {maml_acc:.4f}  (traded for adaptability)")
print(f"  YouTube KNOWN robustness   : {yt_known_acc:.3f}")
print(f"  YouTube NEW  before->after : {yt_new_before:.3f} -> {yt_new_after:.3f}")

In [ ]:
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
payload = {"base_acc": base_acc, "continual_acc": cl_acc, "student_acc": student_acc, "maml_student_acc": maml_acc,
           "yt_known_acc": yt_known_acc, "yt_new_before": yt_new_before, "yt_new_after": yt_new_after,
           "teacher_params": teacher_params, "student_params": student_params,
           "maml_curve": hist_maml["query_accs"]}
with open(f"{cfg.RESULTS_DIR}/ulal_summary.json", "w") as f:
    json.dump(payload, f, indent=2)
print(f"saved -> {cfg.RESULTS_DIR}/ulal_summary.json")